# Lab 3 at your own pace

Build a profile assistant, understand its messages, then add evaluation and one retry.
Read [the walkthrough](lab3-walkthrough.md) beside [the source notebook](../../1_foundations/3_lab3.ipynb).

**Prerequisites:** choose this repository's `.venv` kernel. Familiarity with basic strings and functions helps, but each new concept is explained below.

**How to practice:** predict the output, run one cell with Shift+Enter, explain it, then make the suggested change. Do not rush to Run All until you understand each section.

**Default behavior:** fictional data, simulated replies, no API calls, no credential-file access, no email delivery, no UI server, and no persistent data writes. Optional real calls are at the end and start disabled.

Sections: 1 setup · 2 input text · 3 PDF extraction · 4 prompt · 5 messages · 6 callback/UI · 7 evaluation · 8 feedback · 9 workflow · 10 tools · 11 hosted calls.

## 1. Establish the environment (source cells 3–4)
Imports make existing code available. A client is an object used to request model output later.
This offline section imports no model client and does not load your credentials.
**Predict:** what folder should `ROOT` identify, even if the kernel starts in `docs/exercises`?

In [ ]:
from pathlib import Path
import sys

def find_repo_root():
    # Start with the kernel's working directory, then inspect its parents.
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "1_foundations" / "3_lab3.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Start the kernel somewhere inside ai_agents.")

ROOT = find_repo_root()
print("Repository:", ROOT)
print("Interpreter:", sys.executable)
RUN_HOSTED = False  # Change only in section 11 when you intend provider calls.
RUN_UI = False      # Change only if you want a local chat server.

## 2. Start with understandable input (source cells 5–8)
We use fictional profile text so you can inspect everything without sharing personal data.
`summary` and `linkedin` are ordinary strings. The model will receive them as context later.
**Exercise:** add a skill to the summary, then rerun this cell. What changes downstream?

In [ ]:
name = "Alex Rivera"
summary = "Alex is a Python developer who builds small automation tools."
linkedin = "Alex built a file-organizing script and a simple dashboard."
print(name)
print(summary)
print(linkedin)

**Check:** the name and two strings print. They do not train a model. If you change them, rebuild the prompt in section 4.

The instructor reads summary text with `with open(...)`. Practice that resource-lifetime pattern with an in-memory file, so this exercise writes nothing to disk:

In [ ]:
from io import StringIO

with StringIO(summary) as f:
    summary_copy = f.read()  # Read all the text while the file-like object is open.
print(summary_copy)
print("Closed after with:", f.closed)
assert summary_copy == summary
assert f.closed

## 3. Understand the PDF loop before using a PDF (source cells 5–6)
A page may yield text, an empty string, or `None`. The `if` skips empty/missing text.
**Predict:** will `None` appear in the final profile string?

In [ ]:
page_texts = ["Alex builds automation tools.", None, "Alex knows Python."]
extracted = ""
for text in page_texts:
    if text:                      # Only nonempty values enter this branch.
        extracted += text + "\n"  # Preserve a visible page boundary.
print(extracted)
assert "None" not in extracted

**Check:** two lines, no `None`. A newline makes page boundaries clearer than direct concatenation.

Optional real extraction below is disabled. It reads the source PDF locally and prints counts only; it does not send anything to a model or replace the fictional profile.

In [ ]:
READ_SOURCE_PDF = False
if READ_SOURCE_PDF:
    from pypdf import PdfReader
    pdf_path = ROOT / "1_foundations" / "me" / "linkedin.pdf"
    reader = PdfReader(pdf_path)
    pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    print("Pages:", len(reader.pages), "Characters:", len(pdf_text))
else:
    print("Optional PDF reading skipped.")

## 4. Assemble instructions and evidence (source cells 9–10)
An f-string inserts values at the moment it is built. `\n` means newline.
We use a function so rebuilding the prompt is explicit.
**Predict:** what happens to an already-built prompt if you later change `name`?

In [ ]:
def build_system_prompt(person, summary_text, profile_text):
    return (
        f"You represent {person} on their website. "
        "Answer professionally using only the supplied background. "
        "If the background does not answer a question, say you do not know.\n\n"
        f"## Summary\n{summary_text}\n\n"
        f"## Profile\n{profile_text}"
    )

system_prompt = build_system_prompt(name, summary, linkedin)
print(system_prompt)
assert name in system_prompt and summary in system_prompt

**Answer:** the existing string does not update automatically. Rerun the prompt cell.
**Exercise:** replace the name and rebuild. Explain which part describes behavior and which part supplies evidence. Is a patent claim supported by this profile?

## 5. Build the conversation (source cells 11–12)
A request is a list. Each dictionary has `role` (speaker) and `content` (text).
The order is instructions → old turns → new question. Gradio supplies old turns as `history`.
The helper below keeps only the fields used by this text-only exercise.

In [ ]:
def clean_history(history):
    return [{"role": h["role"], "content": h["content"]} for h in history]

def build_messages(message, history, instructions=None):
    if instructions is None:
        instructions = system_prompt  # Use the current prompt, not an old default value.
    return (
        [{"role": "system", "content": instructions}]
        + clean_history(history)
        + [{"role": "user", "content": message}]
    )

first_request = build_messages("What do you build?", [])
print([m["role"] for m in first_request])
assert len(first_request) == 2

**Expected:** `['system', 'user']`. The empty history adds no messages.
**Predict:** how many messages will a follow-up have after one user/assistant exchange?

In [ ]:
history = [
    {"role": "user", "content": "What do you build?", "extra": "discard me"},
    {"role": "assistant", "content": "I build automation tools."},
]
follow_up = build_messages("Which language do you use?", history)
print([m["role"] for m in follow_up])
print("Count:", len(follow_up))
assert len(follow_up) == 4
assert "extra" not in follow_up[1]

**Expected:** system, user, assistant, user: four messages.
**Exercise:** remove history. Would the new question still have access to the earlier exchange? The profile stays in the system message, but the earlier conversation disappears.

## 6. A callback connects Python to a UI (source cells 11–13)
Defining `chat` does not execute it. Calling it does. Gradio can call it for each turn.
Our fake model returns a deterministic reply; it does not infer or generate like an LLM.

In [ ]:
def fake_model(messages):
    question = messages[-1]["content"]
    return f"Practice reply to: {question}"

def simple_chat(message, history):
    messages = build_messages(message, history)
    reply = fake_model(messages)
    return reply  # Gradio displays the return value, not print output.

print(simple_chat("What do you build?", []))

The real client has a nested response. Practice reading that shape without a provider call:

In [ ]:
from types import SimpleNamespace

response = SimpleNamespace(choices=[
    SimpleNamespace(message=SimpleNamespace(content="I build automation tools."))
])
reply = response.choices[0].message.content
print(reply)
assert reply == "I build automation tools."

**Exercise:** explain `[0]`, `.message`, and `.content` separately.
Optional UI: set `RUN_UI = True` in section 1 and rerun this cell to start a local interface using the fake callback. Stop it with `demo.close()` before rebuilding it.

In [ ]:
if RUN_UI:
    import gradio as gr
    demo = gr.ChatInterface(simple_chat, type="messages", analytics_enabled=False)
    demo.launch(share=False, prevent_thread_lock=True)
else:
    print("UI skipped. Calling simple_chat directly needs no server.")

## 7. Give evaluation a clear shape (source cells 15–22)
`bool` is a True/False decision. `str` is text feedback. Pydantic validates the shape.
It does not prove the evaluator is correct.

In [ ]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

accepted = Evaluation(is_acceptable=True, feedback="Supported by the profile.")
rejected = Evaluation(is_acceptable=False, feedback="Remove the unsupported patent claim.")
print(accepted.model_dump())
print(rejected.feedback)

The evaluator must see the profile, question, conversation, and proposed reply.
**Exercise:** identify each of those inputs in these two prompts.

In [ ]:
evaluator_system_prompt = (
    "Judge whether the answer is professional and supported by this profile. "
    "Reject invented background facts.\n\n"
    + system_prompt
)

def evaluator_user_prompt(reply, message, history):
    return (
        f"Earlier conversation: {clean_history(history)}\n"
        f"Latest question: {message}\n"
        f"Proposed answer: {reply}\n"
        "Return is_acceptable and feedback."
    )

print(evaluator_user_prompt("I hold three patents.", "Do you hold a patent?", []))

This offline evaluator only recognizes our specific bad examples. It makes branches visible; it is not a general quality checker.
**Predict:** will the unsupported patent claim be accepted?

In [ ]:
def evaluate_offline(reply, message, history):
    if "three patents" in reply.lower():
        return Evaluation(is_acceptable=False, feedback="The profile does not mention patents.")
    if "atinlay" in reply.lower():
        return Evaluation(is_acceptable=False, feedback="Answer in ordinary English, not Pig Latin.")
    return Evaluation(is_acceptable=True, feedback="No demonstration failure detected.")

decision = evaluate_offline("I hold three patents.", "Do you hold a patent?", [])
print(decision.model_dump())
assert decision.is_acceptable is False

**Check:** rejection plus a reason. Try an ordinary answer, then try a different invented fact. Notice that the simple rule can miss bad answers. An LLM evaluator can also make mistakes, so judgment still needs verification.

## 8. Feedback becomes input to the next attempt (source cell 23)
The revised prompt preserves the profile and adds the bad draft and its rejection reason.
**Predict:** does this overwrite the original system prompt?

In [ ]:
def revision_prompt(reply, feedback):
    return (
        system_prompt
        + f"\n\n## Previous answer rejected\n{reply}"
        + f"\n\n## Feedback\n{feedback}"
    )

before = system_prompt
updated = revision_prompt("I hold three patents.", decision.feedback)
print(updated)
assert system_prompt == before
assert "three patents" in updated and decision.feedback in updated

**Check:** a new string was created. The original remains unchanged.
**Exercise:** write a revised answer by hand. A good answer admits that the supplied profile does not establish whether Alex holds a patent.

## 9. Put generation → evaluation → one retry together (source cells 24–25)
The instructor deliberately requests Pig Latin for a patent question to trigger rejection.
This reproduction uses fake functions. The code flow is real; the replies are simulated.

In [ ]:
def draft_offline(message, history):
    if "patent" in message.lower():
        return "Iway answerway inway atinlay."  # Deliberately bad demo answer.
    return "I build small Python automation tools."

def rerun_offline(reply, message, history, feedback):
    updated = revision_prompt(reply, feedback)
    # A real writer would receive this prompt via build_messages + an API call.
    assert feedback in updated
    return "My supplied profile does not mention patents, so I cannot confirm that."

def workflow(message, history, draft_fn, evaluate_fn, rerun_fn):
    draft = draft_fn(message, history)
    evaluation = evaluate_fn(draft, message, history)
    final = draft
    retried = False
    if not evaluation.is_acceptable:
        final = rerun_fn(draft, message, history, evaluation.feedback)
        retried = True
    # Like the instructor, this returns one retry without evaluating it again.
    return {"draft": draft, "evaluation": evaluation, "final": final, "retried": retried}

normal = workflow("What do you build?", [], draft_offline, evaluate_offline, rerun_offline)
patent = workflow("Do you hold a patent?", [], draft_offline, evaluate_offline, rerun_offline)
for result in (normal, patent):
    print("Draft:", result["draft"])
    print("Decision:", result["evaluation"].model_dump())
    print("Retried:", result["retried"])
    print("Final:", result["final"], "\n")
assert normal["retried"] is False
assert patent["retried"] is True

**Exercise:** explain the `if` branch without reading the code. An accepted hosted turn needs a writer call and evaluator call; a rejected turn adds one writer call. The final retry is **not** automatically guaranteed to pass.

The instructor redefines `chat`; the newest definition replaces the old one in memory. Our practice functions use distinct names so you can inspect both versions.

In [ ]:
def evaluated_chat(message, history):
    result = workflow(message, history, draft_offline, evaluate_offline, rerun_offline)
    return result["final"]

print(evaluated_chat("Do you hold a patent?", []))

## 10. Email function vs. tool schema (local source cells 27–29)
This extension comes after the main lab. The source writes `emails.txt`; here we store only fictional addresses in memory.
Defining a function, describing its arguments, and executing it are three separate actions.

In [ ]:
practice_emails = []

def record_email_tool(email):
    practice_emails.append(email)  # No file and no email delivery.
    return "email received"

record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Record an email address supplied by the user",
    "parameters": {
        "type": "object",
        "properties": {"email": {"type": "string"}},
        "required": ["email"],
        "additionalProperties": False,  # Python False; JSON spelling would be false.
    },
}
assert practice_emails == []  # Defining the function/schema did not call it.
print(record_email_tool("learner@example.com"))
print(practice_emails)

**Exercise:** restart from this cell and call the function twice. Why are there two addresses? The schema is not connected to model calls here. A later tool lesson supplies schemas to the model and dispatches requested function calls.

## 11. Optional real model calls — disabled by default
Only enable this after the offline exercises make sense. The next cell loads exactly `ROOT / '.environment' / '.env'` **only when `RUN_HOSTED` is True**. It does not print tokens. The file stores plaintext credentials; process variables hold them in memory.

Requires `OPENAI_API_KEY` (writer) and `GOOGLE_API_KEY` (evaluator). Optional `LAB3_WRITER_MODEL` and `LAB3_EVALUATOR_MODEL` override the source's model identifiers. Provider availability and structured parsing are **UNVERIFIED at runtime**.

The optional call below sends fictional profile context and a question to OpenAI and Google. It can incur API costs. It skips email tools and does not launch a UI.

In [ ]:
if RUN_HOSTED:
    import os
    from dotenv import load_dotenv
    from openai import OpenAI

    env_path = ROOT / ".environment" / ".env"
    if not env_path.is_file():
        raise FileNotFoundError(f"Create your local configuration at {env_path}")
    load_dotenv(env_path, override=True)
    for key_name in ("OPENAI_API_KEY", "GOOGLE_API_KEY"):
        if not os.getenv(key_name):
            raise RuntimeError(f"Missing configuration variable: {key_name}")
    writer_model = os.getenv("LAB3_WRITER_MODEL", "gpt-4o-mini")
    evaluator_model = os.getenv("LAB3_EVALUATOR_MODEL", "gemini-2.0-flash")

    def writer_call(messages):
        with OpenAI(api_key=os.environ["OPENAI_API_KEY"], timeout=30, max_retries=0) as client:
            response = client.chat.completions.create(model=writer_model, messages=messages)
            text = response.choices[0].message.content
            if text is None:
                raise RuntimeError("Writer returned no text.")
            return text

    def draft_hosted(message, history):
        return writer_call(build_messages(message, history))

    def evaluate_hosted(reply, message, history):
        with OpenAI(
            api_key=os.environ["GOOGLE_API_KEY"],
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            timeout=30, max_retries=0,
        ) as client:
            response = client.beta.chat.completions.parse(
                model=evaluator_model,
                messages=[
                    {"role": "system", "content": evaluator_system_prompt},
                    {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
                ],
                response_format=Evaluation,
            )
            parsed = response.choices[0].message.parsed
            if parsed is None:
                raise RuntimeError("Evaluator returned no parsed decision; check refusal or compatibility.")
            return parsed

    def rerun_hosted(reply, message, history, feedback):
        return writer_call(build_messages(message, history, revision_prompt(reply, feedback)))

    hosted_result = workflow(
        "Do you hold a patent?", [], draft_hosted, evaluate_hosted, rerun_hosted
    )
    print("Evaluation:", hosted_result["evaluation"].model_dump())
    print("Retried:", hosted_result["retried"])
    print("Reply:", hosted_result["final"])
else:
    print("Hosted calls skipped. No private configuration file was loaded.")

## Explain it back
1. Why are profile context and history separate?
2. Why does changing a variable not rebuild an old string?
3. What connects a Python callback to a chat UI?
4. Does an acceptable Pydantic object guarantee a true answer?
5. What does the writer receive on retry?
6. Is the retry evaluated again in this workflow?
7. Did the tool schema itself save or send an email?

**Answers:** profile supplies background, history supplies old turns; strings were built earlier; `ChatInterface` connects the callback; no, shape is not truth; rejected draft plus feedback; no, there is one unchecked retry; no, the explicit function call recorded a fictional address in memory.

**Next practice:** reconstruct `build_messages` and the `if` branch from memory. Then compare them with source cells 11 and 24. For an extension, add a second evaluation with a fixed retry limit and show what happens when that second evaluation also fails.